In [1]:
# https://huggingface.co/fla-hub/rwkv7-2.9B-g1
# GPU T4 x2

In [2]:
!pip install -q git+https://github.com/fla-org/flash-linear-attention
!pip install -q 'transformers>=4.48.0'

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [3]:
from transformers import AutoModelForCausalLM, AutoTokenizer

In [4]:
model_path = 'fla-hub/rwkv7-2.9B-g1'
tokenizer_path = 'fla-hub/rwkv7-2.9B-g1'

In [5]:
model = AutoModelForCausalLM.from_pretrained(model_path, trust_remote_code=True)
tokenizer = AutoTokenizer.from_pretrained(tokenizer_path, trust_remote_code=True) 
model = model.cuda() # Supported on Nvidia/AMD/Intel eg. model.xpu()

2025-10-19 15:01:21.247620: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1760886081.273520     861 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1760886081.280832     861 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2225: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `ty

In [8]:
prompt = "I have a glass with a no bottom and a sealed top. How can I drink from it?"

messages = [
    {"role": "user", "content": prompt}
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=True  # Default is True, set to False to disable thinking
)

print(text)

model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

generated_ids = model.generate(
    **model_inputs,
    max_new_tokens=2048,
    do_sample=True,
    temperature=1.0,
    top_p=0.3,
    repetition_penalty=1.5
)

generated_ids = [
    output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
]

response = tokenizer.batch_decode(generated_ids, skip_special_tokens=False)[0]

<|rwkv_tokenizer_end_of_text|>User: I have a glass with a no bottom and a sealed top. How can I drink from it?

Assistant: <think


/usr/local/lib/python3.11/dist-packages/fla/ops/rwkv7/fused_recurrent.py:303: UserWarning: Input tensor shape suggests potential format mismatch: seq_len (29) < num_heads (40). This may indicate the inputs were passed in head-first format [B, H, T, ...] when head_first=False was specified. Please verify your input tensor format matches the expected shape [B, T, H, ...].
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/fla/ops/rwkv7/fused_recurrent.py:303: UserWarning: Input tensor shape suggests potential format mismatch: seq_len (29) < num_heads (40). This may indicate the inputs were passed in head-first format [B, H, T, ...] when head_first=False was specified. Please verify your input tensor format matches the expected shape [B, T, H, ...].
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/fla/ops/rwkv7/fused_recurrent.py:303: UserWarning: Input tensor shape suggests potential format mismatch: seq_len (1) < num_heads (40). This may indicate the inputs were passed in 

In [9]:
print(response)

>Okay, so I have this glass that's got no bottom and a sealed top. Hmm, that sounds interesting but also kind of tricky. Let me try to figure out how I can drink from it. First, I need to understand the structure of the glass. It has a top that's sealed, which means there's no opening at the top where liquid would normally go in. So, I guess the only way to get liquid into the glass is through the sides or maybe by some other method.
Wait, but if the bottom is missing, does that mean the glass is just an open container without any base? That might make it unstable. Maybe it's designed to be used on a flat surface or perhaps it's meant to be filled through another part. But since the question is about drinking from it, I think the focus is on getting liquid into the glass somehow.
I remember that sometimes people use straws for drinks, but this glass doesn't have a straw hole. Maybe I can create one myself. If I could find a small piece of material or something flexible, I could poke a 